<a href="https://colab.research.google.com/github/cidadesdofuturo/Cidades-do-Futuro/blob/main/Calculo_indicadores_Inteli_gente_CORRIGIDO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🏙️ Plataforma inteli.gente — Calculador de Níveis de Maturidade
**Metodologia:** Manual de Referência v3.0 (Maio 2023)

Este notebook calcula os níveis de maturidade das quatro dimensões avaliadas
e salva os resultados diretamente na planilha **indicadores**.

**Como usar:**
1. Execute a Célula 1 para instalar dependências
2. Execute a Célula 2 para fazer upload das planilhas
3. Execute a Célula 3 para carregar os dados e escolher o município
4. Execute a Célula 4 para calcular e salvar os resultados

## Célula 1 — Instalar dependências e importar bibliotecas

In [ ]:
# ── Instalar openpyxl caso não esteja disponível ──────────────────────────────
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'openpyxl', '-q'])

# ── Imports ───────────────────────────────────────────────────────────────────
from __future__ import annotations
import re
import pandas as pd
import openpyxl
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
from dataclasses import dataclass, field
from typing import Optional
from pathlib import Path
from google.colab import files
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

print('✅ Dependências carregadas com sucesso!')

✅ Dependências carregadas com sucesso!


## Célula 2 — Upload das planilhas

In [ ]:
print('📂 Faça o upload de DUAS planilhas:')
print('   1️⃣  Respostas.xlsx  (formulário preenchido pelos municípios)')
print('   2️⃣  indicadores.xlsx (planilha de destino dos resultados)')
print()

uploaded = files.upload()

# Detectar arquivos automaticamente
RESPOSTAS_FILE = None
INDICADORES_FILE = None

for fname in uploaded.keys():
    fname_lower = fname.lower()
    if 'resposta' in fname_lower:
        RESPOSTAS_FILE = fname
    elif 'indicador' in fname_lower:
        INDICADORES_FILE = fname

# Fallback: se nomes não batem, usar ordem de upload
if not RESPOSTAS_FILE or not INDICADORES_FILE:
    keys = list(uploaded.keys())
    if len(keys) >= 2:
        if not RESPOSTAS_FILE:
            RESPOSTAS_FILE = keys[0]
        if not INDICADORES_FILE:
            INDICADORES_FILE = keys[1]

if RESPOSTAS_FILE and INDICADORES_FILE:
    print(f'\n✅ Respostas  → {RESPOSTAS_FILE}')
    print(f'✅ Indicadores → {INDICADORES_FILE}')
else:
    print('⚠️  Não foi possível identificar os arquivos. Verifique o upload.')

📂 Faça o upload de DUAS planilhas:
   1️⃣  Respostas.xlsx  (formulário preenchido pelos municípios)
   2️⃣  indicadores.xlsx (planilha de destino dos resultados)



Saving indicadores.xlsx to indicadores (1).xlsx
Saving Respostas.xlsx to Respostas (1).xlsx

✅ Respostas  → Respostas (1).xlsx
✅ Indicadores → indicadores (1).xlsx


## Célula 3 — Carregar dados e selecionar município

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 1. ESTRUTURAS DE DADOS
# ══════════════════════════════════════════════════════════════════════════════

@dataclass
class RangeNivel:
    nivel: int
    min_pts: float
    max_pts: float
    acompanha_topico: bool = False

@dataclass
class Indicador:
    codigo: str
    nome: str
    topico: str
    dimensao: str
    tipo: str
    relevancia: str
    alternativas: dict
    ranges: list
    col_idx: int           # índice (base-0) da coluna no Respostas.xlsx
    modo_calculo: str = 'soma'

    @property
    def peso_relevancia(self):
        return {'Alta': 3, 'Média': 2, 'Baixa': 1}[self.relevancia]

@dataclass
class ResultadoIndicador:
    indicador: Indicador
    resposta_raw: str
    letras_selecionadas: list
    pontuacao_bruta: float
    nivel: int
    acompanha_topico: bool = False

@dataclass
class ResultadoTopico:
    topico: str
    indicadores: list = field(default_factory=list)

    @property
    def nivel_medio(self):
        if not self.indicadores:
            return 0.0
        # Média ponderada pelos pesos de relevância (Alta=3, Média=2, Baixa=1)
        soma_pond = sum(r.nivel * r.indicador.peso_relevancia for r in self.indicadores)
        soma_pesos = sum(r.indicador.peso_relevancia for r in self.indicadores)
        return soma_pond / soma_pesos

    @property
    def nivel_arredondado(self):
        return round(self.nivel_medio)

@dataclass
class ResultadoDimensao:
    dimensao: str
    topicos: dict = field(default_factory=dict)

    @property
    def nivel_medio(self):
        niveis = [t.nivel_medio for t in self.topicos.values() if t.indicadores]
        return sum(niveis) / len(niveis) if niveis else 0.0

    @property
    def nivel_arredondado(self):
        return round(self.nivel_medio)

# ══════════════════════════════════════════════════════════════════════════════
# 2. HELPER FUNCTIONS
# ══════════════════════════════════════════════════════════════════════════════

def _r(*args):
    result = []
    for t in args:
        nivel, mn, mx = t[0], t[1], t[2]
        at = t[3] if len(t) > 3 else False
        result.append(RangeNivel(nivel, mn, mx, at))
    return result

def parsear_resposta(resposta_raw):
    if not resposta_raw or pd.isna(resposta_raw):
        return []
    texto = str(resposta_raw).strip()
    letras = re.findall(r'\b([A-Z])(?=\s*[-–])', texto)
    vistos, resultado = set(), []
    for l in letras:
        if l not in vistos:
            vistos.add(l)
            resultado.append(l)
    return resultado

def calcular_pontuacao(indicador, letras):
    if not letras:
        return 0.0
    if indicador.modo_calculo == 'soma':
        return sum(indicador.alternativas.get(l, 0) for l in letras)
    elif indicador.modo_calculo == 'unico':
        return max((indicador.alternativas.get(l, 0) for l in letras), default=0.0)
    raise ValueError(f'Modo desconhecido: {indicador.modo_calculo}')

def pontuacao_para_nivel(indicador, pontuacao):
    """Retorna (nivel_base, acompanha_topico).

    Quando acompanha_topico=True o nível aqui é provisório;
    o nível definitivo é sobrescrito em processar_municipio()
    após calcular o nível médio do tópico com os indicadores fixos.
    """
    candidatos = [r for r in indicador.ranges if r.min_pts <= pontuacao <= r.max_pts]
    if not candidatos:
        r = indicador.ranges[0] if pontuacao < indicador.ranges[0].min_pts else indicador.ranges[-1]
    else:
        # Prefere ranges SEM acompanha_topico; se todos têm AT usa o primeiro candidato
        sem_at = [c for c in candidatos if not c.acompanha_topico]
        r = sem_at[0] if sem_at else candidatos[0]
    return r.nivel, r.acompanha_topico

def calcular_indicador(indicador, row):
    resposta_raw = row.iloc[indicador.col_idx]
    letras = parsear_resposta(resposta_raw)
    pontuacao = calcular_pontuacao(indicador, letras)
    nivel, acompanha = pontuacao_para_nivel(indicador, pontuacao)
    return ResultadoIndicador(
        indicador=indicador,
        resposta_raw=str(resposta_raw),
        letras_selecionadas=letras,
        pontuacao_bruta=pontuacao,
        nivel=nivel,
        acompanha_topico=acompanha,
    )

# ══════════════════════════════════════════════════════════════════════════════
# 3. DEFINIÇÃO DOS INDICADORES — TODAS AS DIMENSÕES
#    Fonte: Manual inteli.gente v3.0, Seção 8
#    col_idx = índice (base-0) da coluna na planilha Respostas
# ══════════════════════════════════════════════════════════════════════════════

TODOS_INDICADORES = [

    # ── DIMENSÃO: CAPACIDADES INSTITUCIONAIS ──────────────────────────────────
    # Tópico: Estratégia
    Indicador('CI_EST_01', 'Incorporação de TICs - Planejamento',
              'Estratégia', 'Capacidades Institucionais',
              'Principal', 'Alta',
              {'A': 2, 'B': 3, 'C': 1, 'D': 1, 'E': 3},
              _r((1,0,0),(2,1,2),(3,3,4),(4,5,6),(5,7,7),(6,8,9),(7,10,10)),
              col_idx=14),

    Indicador('CI_EST_02', 'Planejamento Estratégico para Transformação Digital',
              'Estratégia', 'Capacidades Institucionais',
              'Principal', 'Alta',
              {'A': 2, 'B': 2, 'C': 2, 'D': 2},
              _r((1,0,0),(2,2,2),(3,4,4),(4,6,6),(5,8,8,True),(6,8,8,True),(7,8,8,True)),
              col_idx=15),

    Indicador('CI_EST_03', 'Governança Colaborativa - Responsáveis',
              'Estratégia', 'Capacidades Institucionais',
              'Principal', 'Alta',
              {l: 1 for l in 'ABCDEFGHIJKLM'},
              _r((1,0,1),(2,2,3),(3,4,6),(4,7,8),(5,9,10),(6,11,12),(7,13,13)),
              col_idx=16),

    # Tópico: Infraestrutura de Hw e Sw
    Indicador('CI_INF_01', 'Governança de TI - Práticas',
              'Infraestrutura de Hw e Sw', 'Capacidades Institucionais',
              'Principal', 'Média',
              {l: 1 for l in 'ABCDE'},
              _r((1,0,0,True),(2,0,0,True),(3,1,1),(4,2,2),(5,3,3),(6,4,4),(7,5,5)),
              col_idx=17),

    Indicador('CI_INF_02', 'Infraestrutura de Hw e Sw - Armazenamento',
              'Infraestrutura de Hw e Sw', 'Capacidades Institucionais',
              'Principal', 'Média',
              {'A': 3, 'B': 3, 'C': 3, 'D': 2, 'E': 1, 'F': 1, 'G': 1},
              _r((1,0,1),(2,2,2),(3,3,4),(4,5,6),(5,7,8),(6,9,10),(7,11,14)),
              col_idx=18),

    # Tópico: Serviços e Aplicações
    Indicador('CI_SRV_01', 'Gestão Integrada de Dados',
              'Serviços e Aplicações', 'Capacidades Institucionais',
              'Principal', 'Média',
              {'A': 1, 'B': 2, 'C': 3, 'D': 3, 'E': 3},
              _r((1,0,0,True),(2,0,0,True),(3,1,1),(4,2,2,True),(5,2,2,True),(6,3,3,True),(7,3,3,True)),
              col_idx=19, modo_calculo='unico'),

    # Tópico: Dados Abertos
    Indicador('CI_DAD_01', 'Segurança dos Dados - Práticas',
              'Dados Abertos', 'Capacidades Institucionais',
              'Principal', 'Alta',
              {l: 1 for l in 'ABCDEFGH'},
              _r((1,0,0),(2,1,1),(3,2,3),(4,4,5),(5,6,6),(6,7,7),(7,8,8)),
              col_idx=20),

    # Tópico: Monitoramento
    Indicador('CI_MON_01', 'Segurança de Políticas Públicas - Monitoramento',
              'Monitoramento', 'Capacidades Institucionais',
              'Principal', 'Média',
              {l: 1 for l in 'ABCD'},
              _r((1,0,0),(2,1,1),(3,2,2),(4,3,3),(5,4,4,True),(6,4,4,True),(7,4,4,True)),
              col_idx=21),

    Indicador('CI_MON_02', 'Transparência - Monitoramento',
              'Monitoramento', 'Capacidades Institucionais',
              'Principal', 'Média',
              {'A': 1, 'B': 2, 'C': 3},
              _r((1,0,0),(2,1,1,True),(3,1,1,True),(4,2,2,True),(5,2,2,True),(6,3,3,True),(7,3,3,True)),
              col_idx=22, modo_calculo='unico'),

    # CI_MON_03 — Percepção dos Serviços Públicos (Manual pág. 142)
    # 0 pt -> níveis 1-3 acompanha tópico;  1 pt -> níveis 4-7 acompanha tópico
    # Fonte IBGE/MUNIC: MTIC1218 (pesquisa satisfação serviço público)
    Indicador('CI_MON_03', 'Percepção dos Serviços Públicos',
              'Monitoramento', 'Capacidades Institucionais',
              'Principal', 'Média',
              {'A': 1},
              _r((1,0,0,True),(2,0,0,True),(3,0,0,True),(4,1,1,True),(5,1,1,True),(6,1,1,True),(7,1,1,True)),
              col_idx=23, modo_calculo='unico'),

    # ── DIMENSÃO: ECONÔMICA ────────────────────────────────────────────────────
    # Tópico: Mobilidade / Transporte
    # EC_TRN_01 — Manual pág. 36: níveis 1-2=0 AT; 3=1pt; 4=2pt; 5=3pt; 6=4pt; 7=5pt
    # Fórmula: A*1+B*1+C*1+D*1
    Indicador('EC_TRN_01', 'Serviços de compartilhamento de viagens',
              'Transporte', 'Econômica',
              'Principal', 'Baixa',
              {'A': 1, 'B': 1, 'C': 1, 'D': 1},
              _r((1,0,0,True),(2,0,0,True),(3,1,1),(4,2,2),(5,3,3),(6,4,4),(7,5,5)),
              col_idx=23),

    # EC_TRN_02 — Manual pág. 37: 1-2=0 AT; 3=1pt; 4=2pt; 5=3pt; 6-7=4pt AT
    # Fórmula: A*1+B*1+C*1+D*1
    Indicador('EC_TRN_02', 'Serviço de informações de transporte público em tempo real',
              'Transporte', 'Econômica',
              'Principal', 'Baixa',
              {'A': 1, 'B': 1, 'C': 1, 'D': 1},
              _r((1,0,0,True),(2,0,0,True),(3,1,1),(4,2,2),(5,3,3),(6,4,4,True),(7,4,4,True)),
              col_idx=24),

    # EC_TRN_03 — Manual pág. 64: 1=0; 2-3=1pt AT; 4-5=2pt AT; 6-7=3pt AT
    # Fórmula: A*1 OU B*2 OU C*3
    Indicador('EC_TRN_03', 'Plataforma integrada para cidade inteligente',
              'Transporte', 'Econômica',
              'Principal', 'Baixa',
              {'A': 1, 'B': 2, 'C': 3},
              _r((1,0,0),(2,1,1,True),(3,1,1,True),(4,2,2,True),(5,2,2,True),(6,3,3,True),(7,3,3,True)),
              col_idx=25, modo_calculo='unico'),

    # EC_SEG_01 — Manual pág. 105: 1-2=0 AT; 3=1pt; 4=2pt; 5=3pt; 6=4pt; 7=5pt
    # Fórmula: A*1+B*1+C*1+D*1+E*1
    Indicador('EC_SEG_01', 'Soluções em monitoramento para segurança pública',
              'Segurança Pública', 'Econômica',
              'Principal', 'Baixa',
              {l: 1 for l in 'ABCDE'},
              _r((1,0,0,True),(2,0,0,True),(3,1,1),(4,2,2),(5,3,3),(6,4,4),(7,5,5)),
              col_idx=26),

    # ── DIMENSÃO: SOCIOCULTURAL ───────────────────────────────────────────────
    # Tópico: Cultura
    # SC_CUL_01 — Manual pág. 69: 1-2=0 AT; 3=1-2; 4=3; 5=4; 6=5; 7=6
    # Fórmula: A*1+B*1+C*1+D*1+E*1+F*1
    Indicador('SC_CUL_01', 'Serviços on-line para promoção de cultura',
              'Cultura', 'Sociocultural',
              'Principal', 'Baixa',
              {l: 1 for l in 'ABCDEF'},
              _r((1,0,0,True),(2,0,0,True),(3,1,2),(4,3,3),(5,4,4),(6,5,5),(7,6,6)),
              col_idx=27),

    # SC_CUL_02 — Manual pág. 71: 1-2=0 AT; 3=1-3; 4=4-5; 5=6; 6=7; 7=8
    # Fórmula: A*1+B*1+C*1+D*1+E*1+F*1+G*1+H*1
    Indicador('SC_CUL_02', 'Serviços culturais on-line oferecidos para a população',
              'Cultura', 'Sociocultural',
              'Principal', 'Baixa',
              {l: 1 for l in 'ABCDEFGH'},
              _r((1,0,0,True),(2,0,0,True),(3,1,3),(4,4,5),(5,6,6),(6,7,7),(7,8,8)),
              col_idx=28),

    # Tópico: Saúde
    # SC_SAU_01 — Manual pág. 97: 1-2=0 AT; 3=1pt; 4=2-3pt; 5=4-5pt; 6=6-8pt; 7=9pt
    # Fórmula: A*1+B*1+C*2+D*1+E*2+F*2
    Indicador('SC_SAU_01', 'Serviços de telemedicina ou telessaúde',
              'Saúde', 'Sociocultural',
              'Principal', 'Baixa',
              {'A': 1, 'B': 1, 'C': 2, 'D': 1, 'E': 2, 'F': 2},
              _r((1,0,0,True),(2,0,0,True),(3,1,1),(4,2,3),(5,4,5),(6,6,8),(7,9,9)),
              col_idx=29),

    # SC_SAU_02 — Manual pág. 100: 1=0; 2-3=1pt AT; 4-7=2pt AT
    # Fórmula: A*1 OU B*2
    Indicador('SC_SAU_02', 'Prontuário eletrônico',
              'Saúde', 'Sociocultural',
              'Principal', 'Baixa',
              {'A': 1, 'B': 2},
              _r((1,0,0),(2,1,1,True),(3,1,1,True),(4,2,2,True),(5,2,2,True),(6,2,2,True),(7,2,2,True)),
              col_idx=30, modo_calculo='unico'),

    # SC_SAU_03 — Manual pág. 101: 1-2=0 AT; 3=1-3; 4=4-5; 5=6; 6=7; 7=8
    # Fórmula: A*1+B*1+C*2+D*2+E*2
    Indicador('SC_SAU_03', 'Serviços on-line de saúde oferecidos aos pacientes',
              'Saúde', 'Sociocultural',
              'Principal', 'Baixa',
              {'A': 1, 'B': 1, 'C': 2, 'D': 2, 'E': 2},
              _r((1,0,0,True),(2,0,0,True),(3,1,3),(4,4,5),(5,6,6),(6,7,7),(7,8,8)),
              col_idx=31),

    # Tópico: Inclusão Digital
    # SC_INC_01 — Manual pág. 88: 1-2=0 AT; 3=1pt; 4=2pt; 5=3pt; 6=4pt; 7=5pt
    # Fórmula: A*1+B*1+C*1+D*1+E*1
    Indicador('SC_INC_01', 'Promoção de inclusão digital',
              'Inclusão Digital', 'Sociocultural',
              'Principal', 'Alta',
              {l: 1 for l in 'ABCDE'},
              _r((1,0,0,True),(2,0,0,True),(3,1,1),(4,2,2),(5,3,3),(6,4,4),(7,5,5)),
              col_idx=32),

    # ── DIMENSÃO: MEIO AMBIENTE ───────────────────────────────────────────────
    # Tópico: Energia
    # MA_ENE_01 — Manual pág. 118: 1=0; 2=1pt; 3=2pt; 4=3-4pt; 5=5-7pt; 6=8pt; 7=9pt (sem AT)
    # Fórmula: A*1+B*3+C*2+D*3
    Indicador('MA_ENE_01', 'Soluções inteligentes para gestão do consumo de energia',
              'Energia', 'Meio Ambiente',
              'Principal', 'Baixa',
              {'A': 1, 'B': 3, 'C': 2, 'D': 3},
              _r((1,0,0),(2,1,1),(3,2,2),(4,3,4),(5,5,7),(6,8,8),(7,9,9)),
              col_idx=33),

    # Tópico: Resíduos Sólidos
    # MA_RES_01 — Manual pág. 125: 1-2=0 AT; 3=1pt; 4=2pt; 5=3pt; 6-7=4pt AT
    # Fórmula: A*1+B*1+C*1+D*1
    Indicador('MA_RES_01', 'Soluções para otimização da coleta de resíduos',
              'Resíduos Sólidos', 'Meio Ambiente',
              'Principal', 'Baixa',
              {l: 1 for l in 'ABCD'},
              _r((1,0,0,True),(2,0,0,True),(3,1,1),(4,2,2),(5,3,3),(6,4,4,True),(7,4,4,True)),
              col_idx=34),

    # Tópico: Água e Esgoto
    # MA_AGU_01 — Manual pág. 113: 1=0; 2=1pt; 3=2pt; 4=3pt; 5=4pt; 6=5pt; 7=6pt (sem AT)
    # Fórmula: A*1+B*3+C*2
    Indicador('MA_AGU_01', 'Soluções inteligentes para gestão da distribuição de água',
              'Água e Esgoto', 'Meio Ambiente',
              'Principal', 'Baixa',
              {'A': 1, 'B': 3, 'C': 2},
              _r((1,0,0),(2,1,1),(3,2,2),(4,3,3),(5,4,4),(6,5,5),(7,6,6)),
              col_idx=35),

    # Tópico: Telegestão Iluminação
    # MA_ILU_01 — Manual pág. 119: 1=0; 2-4=1pt AT; 5-7=2pt AT
    # Fórmula: A*1 OU B*2
    Indicador('MA_ILU_01', 'Soluções para telegestão da iluminação pública',
              'Energia', 'Meio Ambiente',
              'Principal', 'Baixa',
              {'A': 1, 'B': 2},
              _r((1,0,0),(2,1,1,True),(3,1,1,True),(4,1,1,True),(5,2,2,True),(6,2,2,True),(7,2,2,True)),
              col_idx=36, modo_calculo='unico'),

    # Tópico: Qualidade do Ar
    # MA_AR_01 — Manual pág. 120: soluções monitoramento GEE e qualidade do ar
    # 1=0; 2=1-2pt; 3=3-4pt; 4=5-6pt; 5=7-9pt; 6=10-13pt; 7=14-15pt (sem AT)
    # Fórmula: A*2+B*2+C*2+D*1+E*1+F*1+G*1+H*1+I*1+J*1+K*1+L*1
    Indicador('MA_AR_01', 'Monitoramento de emissões de gases de efeito estufa',
              'Qualidade do Ar', 'Meio Ambiente',
              'Principal', 'Baixa',
              {'A': 2, 'B': 2, 'C': 2, 'D': 1, 'E': 1, 'F': 1, 'G': 1, 'H': 1, 'I': 1, 'J': 1, 'K': 1, 'L': 1},
              _r((1,0,0),(2,1,2),(3,3,4),(4,5,6),(5,7,9),(6,10,13),(7,14,15)),
              col_idx=37),

    # MA_AR_02 — Manual pág. 122: 1=0; 2=1pt; 3=2pt; 4=3pt; 5-7=4pt AT
    # Fórmula: A*1+B*1+C*1+D*1
    Indicador('MA_AR_02', 'Monitoramento da qualidade do ar',
              'Qualidade do Ar', 'Meio Ambiente',
              'Principal', 'Baixa',
              {l: 1 for l in 'ABCD'},
              _r((1,0,0),(2,1,1),(3,2,2),(4,3,3),(5,4,4,True),(6,4,4,True),(7,4,4,True)),
              col_idx=38),
]

# ══════════════════════════════════════════════════════════════════════════════
# 4. CARREGAR PLANILHAS E LISTAR MUNICÍPIOS
# ══════════════════════════════════════════════════════════════════════════════

df_resp = pd.read_excel(RESPOSTAS_FILE, header=0)
df_ind  = pd.read_excel(INDICADORES_FILE, header=0)

# Municípios disponíveis no formulário de respostas
municipios_resp = []
for i, row in df_resp.iterrows():
    nome = str(row.iloc[2]).strip()   # coluna 2 = 'Qual município você representa?'
    # Verificar se tem pelo menos algumas respostas CI
    tem_respostas = any(pd.notna(row.iloc[j]) for j in range(14, 23))
    status = '✅' if tem_respostas else '⚠️ sem respostas'
    municipios_resp.append((i, nome, status))

print('📋 Municípios encontrados na planilha Respostas:')
print()
for idx, nome, status in municipios_resp:
    print(f'   [{idx}] {nome}  {status}')

print()
print(f'📊 Total de municípios na planilha Indicadores: {len(df_ind)}')
print()

# ── Widget de seleção ─────────────────────────────────────────────────────────
opcoes = [f'[{i}] {nome}' for i, nome, _ in municipios_resp]

select_widget = widgets.Select(
    options=opcoes,
    description='Município:',
    layout=widgets.Layout(width='500px', height='200px')
)

print('👇 Selecione o município para calcular:')
display(select_widget)
print()
print('✅ Após selecionar, execute a Célula 4.')


📋 Municípios encontrados na planilha Respostas:

   [0] SACRAMENTO /MG  ⚠️ sem respostas
   [1] Brazópolis  ⚠️ sem respostas
   [2] Conceição das Alagoas  ⚠️ sem respostas
   [3] Nova Era  ⚠️ sem respostas
   [4] Iraí de Minas  ⚠️ sem respostas
   [5] Capitólio - MG  ⚠️ sem respostas
   [6] teste  ✅
   [7] Sete Lagoas  ✅
   [8] Janaúba MG  ✅
   [9] Carmo do Cajuru - Mg  ✅
   [10] padre carvalho-MG  ✅
   [11] Córrego Do Bom Jesus  ✅
   [12] Abaeté  ✅
   [13] Mariana MG  ✅
   [14] Cataguases  ✅
   [15] Porteirinha  ✅
   [16] Salinas/MG  ✅
   [17] Sarzedo  ✅
   [18] Planura/MG  ✅
   [19] Bonito de Minas  ✅
   [20] Chácara  ✅

📊 Total de municípios na planilha Indicadores: 853

👇 Selecione o município para calcular:


Select(description='Município:', layout=Layout(height='200px', width='500px'), options=('[0] SACRAMENTO /MG', …


✅ Após selecionar, execute a Célula 4.


## Célula 4 — Calcular indicadores e salvar na planilha

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# FUNÇÕES DE CÁLCULO
# ══════════════════════════════════════════════════════════════════════════════

def processar_municipio(row_resp, indicadores):
    """Processa uma linha do formulário e retorna dict de dimensões.

    Passo 1 - calcula cada indicador individualmente.
    Passo 2 - para indicadores com acompanha_topico=True substitui o
              nível provisório pelo nível médio arredondado do tópico,
              calculado apenas com os indicadores que NÃO acompanham
              o tópico. Se TODOS os indicadores do tópico acompanham
              o tópico (ex.: Percepção dos Serviços Públicos), usa a
              média provisória de todos como estimativa de base.
    """
    dimensoes = {}

    # ── Passo 1: calcular todos os indicadores ────────────────────
    for ind in indicadores:
        dim = ind.dimensao
        top = ind.topico
        if dim not in dimensoes:
            dimensoes[dim] = ResultadoDimensao(dimensao=dim)
        if top not in dimensoes[dim].topicos:
            dimensoes[dim].topicos[top] = ResultadoTopico(topico=top)

        ri = calcular_indicador(ind, row_resp)
        dimensoes[dim].topicos[top].indicadores.append(ri)

    # ── Passo 2: resolver nível do tópico para indicadores AT ─────
    for res_dim in dimensoes.values():
        for res_top in res_dim.topicos.values():
            indicadores_at    = [ri for ri in res_top.indicadores if ri.acompanha_topico]
            indicadores_fixos = [ri for ri in res_top.indicadores if not ri.acompanha_topico]

            if not indicadores_at:
                continue  # nenhum indicador AT neste tópico

            # Base: média ponderada dos indicadores fixos; se todos são AT, usa todos
            base = indicadores_fixos if indicadores_fixos else res_top.indicadores
            soma_p = sum(ri.nivel * ri.indicador.peso_relevancia for ri in base)
            soma_w = sum(ri.indicador.peso_relevancia for ri in base)
            media = soma_p / soma_w if soma_w else 1
            nivel_topico = int(media + 0.5)

            for ri in indicadores_at:
                ri.nivel = nivel_topico

    return dimensoes


# ── Mapeamento: nome do indicador → coluna no indicadores.xlsx ────────────────
# Cada entrada: (col_valor, col_nivel)  ← índices base-0
MAPA_COLS_IND = {
    # Capacidades Institucionais — indicadores calculados pelo formulário
    'Governança Colaborativa - Responsáveis':            (205, 206),
    'Incorporação de TICs - Planejamento':               (207, 208),
    'Planejamento Estratégico para Transformação Digital':(209, 210),
    'Governança de TI - Práticas':                       (212, 213),
    'Infraestrutura de Hw e Sw - Armazenamento':         (214, 215),
    'Gestão Integrada de Dados':                         (217, 218),
    'Segurança de Políticas Públicas - Monitoramento':   (224, 225),
    'Transparência - Monitoramento':                     (228, 229),
    'Segurança dos Dados - Práticas':                    (235, 236),
    # Econômica — indicadores calculados pelo formulário
    'Serviços de compartilhamento de viagens':           (52, 53),
    'Serviço de informações de transporte público em tempo real': (54, 55),
    'Plataforma integrada para cidade inteligente':      (94, 95),
    'Soluções em monitoramento para segurança pública':  (147, 148),
    # Sociocultural — indicadores calculados pelo formulário
    'Serviços on-line para promoção de cultura':         (127, 128),
    'Serviços culturais on-line oferecidos para a população': (129, 130),
    'Serviços de telemedicina ou telessaúde':            (132, 133),
    'Prontuário eletrônico':                             (138, 139),
    'Serviços on-line de saúde oferecidos aos pacientes':(140, 141),
    'Promoção de inclusão digital':                      (159, 160),
    # Meio Ambiente — indicadores calculados pelo formulário
    'Soluções inteligentes para gestão do consumo de energia': (199, 200),
    'Soluções para otimização da coleta de resíduos':    (188, 189),
    'Soluções inteligentes para gestão da distribuição de água': (179, 180),
    'Soluções para telegestão da iluminação pública':    (201, 202),
    'Monitoramento de emissões de gases de efeito estufa': (194, 195),
    'Monitoramento da qualidade do ar':                  (196, 197),
}

# ── Mapeamento de tópicos por dimensão: col_nivel_topico → [cols_N_M_indicadores] ──
# Inclui TODOS os indicadores N M da planilha (calculados e pré-existentes)
# Usado para recalcular o nível do tópico considerando TODOS os seus indicadores
MAPA_TOPICOS_IND = {
    # Capacidades Institucionais
    211: [206, 208, 210],          # Estratégia
    216: [213, 215],               # Infraestrutura de Hw e Sw
    223: [218, 220, 222],          # Serviços e Aplicações (inclui col 220 e 222 pré-existentes)
    230: [225, 227, 229],          # Monitoramento (inclui col 227 pré-existente)
    237: [232, 234, 236],          # Dados Abertos (inclui cols 232 e 234 pré-existentes)
    # Econômica
    37:  [32, 34, 36],             # Água e esgoto
    42:  [39, 41],                 # Resíduos sólidos
    49:  [44, 46, 48],             # Habitação
    62:  [51, 53, 55, 57, 59, 61], # Transporte (inclui vários pré-existentes)
    65:  [64],                     # Urbanização vias públicas
    80:  [67, 69, 71, 73, 75, 77, 79], # Infraestrutura de conectividade
    89:  [82, 84, 86, 88],         # Inovação
    96:  [91, 93, 95],             # Sistemas e tecnologia para gestão urbana
    99:  [98],                     # Serviços on-line da prefeitura
    102: [101],                    # Dados abertos
    # Sociocultural
    122: [105, 107, 109, 111, 113, 115, 117, 119, 121], # Educação
    131: [124, 126, 128, 130],     # Cultura
    146: [133, 135, 137, 139, 141, 143, 145], # Saúde
    153: [148, 150, 152],          # Segurança Pública
    158: [155, 157],               # Gestão de desastres
    163: [160, 162],               # Inclusão digital
    168: [165, 167],               # Inclusão social
    173: [170, 172],               # Participação pública
    # Meio Ambiente
    185: [176, 178, 180, 182, 184], # Água e esgoto
    190: [187, 189],               # Resíduos sólidos
    193: [192],                    # Áreas verdes
    198: [195, 197],               # Qualidade do ar
    203: [200, 202],               # Energia
}

# ── Pesos de relevância por coluna N M (Alta=3, Média=2, Baixa=1) ──────────────
# Ordem correspondente às listas em MAPA_TOPICOS_IND
MAPA_PESOS_IND = {
    # Capacidades Institucionais
    211: [3, 3, 3],          # Estratégia: Alta, Alta, Alta
    216: [2, 2],             # Infraestrutura de Hw e Sw: Média, Média
    223: [2, 2, 2],          # Serviços e Aplicações: Média, Média, Média
    230: [2, 2, 2],          # Monitoramento: Média, Média, Média
    237: [3, 3, 3],          # Dados Abertos: Alta, Alta, Alta
    # Econômica
    37:  [3, 3, 3],          # Água e esgoto: Alta, Alta, Alta
    42:  [3, 2],             # Resíduos sólidos: Alta, Média
    49:  [3, 3, 3],          # Habitação: Alta, Alta, Alta
    62:  [2, 1, 1, 2, 2, 2], # Transporte: Média, Baixa(EC_TRN_01), Baixa(EC_TRN_02), Média, Média, Média
    65:  [3],               # Urbanização vias públicas: Alta
    80:  [3, 3, 3, 3, 3, 3, 3], # Infraestrutura de conectividade: Alta x7
    89:  [2, 2, 2, 2],       # Inovação: Média x4
    96:  [2, 2, 1],          # Sistemas e tecnologia: Média, Média, Baixa(EC_TRN_03)
    99:  [2],               # Serviços on-line: Média
    102: [2],               # Dados abertos: Média
    # Sociocultural
    122: [3, 3, 3, 3, 3, 3, 3, 3, 3], # Educação: Alta x9
    131: [2, 2, 1, 1],       # Cultura: Média, Média, Baixa(SC_CUL_01), Baixa(SC_CUL_02)
    146: [1, 2, 2, 1, 1, 2, 2], # Saúde: Baixa(SC_SAU_01), Média, Média, Baixa(SC_SAU_02), Baixa(SC_SAU_03), Média, Média
    153: [1, 2, 2],          # Segurança Pública: Baixa(EC_SEG_01), Média, Média
    158: [2, 2],             # Gestão de desastres: Média x2
    163: [3, 2],             # Inclusão digital: Alta(SC_INC_01), Média
    168: [2, 2],             # Inclusão social: Média x2
    173: [2, 2],             # Participação pública: Média x2
    # Meio Ambiente
    185: [3, 3, 1, 3, 3],    # Água e esgoto MA: Alta, Alta, Baixa, Alta, Alta
    190: [3, 1],             # Resíduos sólidos MA: Alta, Baixa(MA_RES_01)
    193: [3],               # Áreas verdes: Alta
    198: [1, 1],             # Qualidade do ar: Baixa(MA_AR_01), Baixa(MA_AR_02)
    203: [1, 1],             # Energia: Baixa(MA_ENE_01), Baixa(MA_ILU_01)
}

# ── Tópicos por dimensão (col_nivel_dimensao → [cols_nivel_topico]) ──
MAPA_DIMENSOES_IND = {
    204: [211, 216, 223, 230, 237],          # Capacidades Institucionais
    30:  [37, 42, 49, 62, 65, 80, 89, 96, 99, 102],  # Econômica
    103: [122, 131, 146, 153, 158, 163, 168, 173],   # Sociocultural
    174: [185, 190, 193, 198, 203],          # Meio Ambiente
}


def buscar_linha_municipio(df_ind, nome_municipio):
    """Busca a linha do município na planilha indicadores.

    Prioridade:
      1. Igualdade exata (após normalização) → retorna imediatamente.
      2. Match parcial (substring) → guardado como candidato de fallback.

    Isso evita que "Porteirinha" case com "Nova Porteirinha" antes de
    encontrar a linha correta.
    """
    def normalizar(s):
        s = str(s).strip().lower()
        s = re.sub(r'[\s\-/]*(mg|minas gerais)\s*$', '', s).strip()
        return s

    nome_norm = normalizar(nome_municipio)
    candidato_parcial = None

    for i, row in df_ind.iterrows():
        mun = normalizar(row.iloc[1])

        if mun == nome_norm:
            return i  # match exato → para aqui

        if candidato_parcial is None and (nome_norm in mun or mun in nome_norm):
            candidato_parcial = i  # guarda como fallback

    return candidato_parcial


def salvar_resultados(dimensoes, municipio_nome, df_ind, arquivo_ind):
    """Salva os resultados calculados na planilha indicadores,
    recalculando tópicos e dimensões usando TODOS os indicadores (formulário + planilha)."""
    linha_idx = buscar_linha_municipio(df_ind, municipio_nome)

    if linha_idx is None:
        print(f'⚠️  Município "{municipio_nome}" não encontrado na planilha indicadores.')
        print('    Verifique se o nome está correto ou adicione o município manualmente.')
        return False

    # Abrir planilha com openpyxl para preservar formatação
    wb = openpyxl.load_workbook(arquivo_ind)
    ws = wb.active
    excel_row = linha_idx + 2   # +1 cabeçalho, +1 base-1

    def get_cell(col_0based):
        """Lê o valor atual de uma célula na planilha."""
        val = ws.cell(row=excel_row, column=col_0based + 1).value
        try:
            v = float(val)
            return v if v > 0 else None
        except (TypeError, ValueError):
            return None

    def set_cell(col_0based, value):
        if col_0based is None or value is None:
            return
        ws.cell(row=excel_row, column=col_0based + 1, value=value)

    # ── 1. Salvar indicadores calculados pelo formulário ──────────────────────
    for dim_nome, res_dim in dimensoes.items():
        for top_nome, res_top in res_dim.topicos.items():
            for ri in res_top.indicadores:
                mapa_ind = MAPA_COLS_IND.get(ri.indicador.nome)
                if mapa_ind:
                    set_cell(mapa_ind[0], ri.pontuacao_bruta if mapa_ind[0] else None)
                    set_cell(mapa_ind[1], ri.nivel)

    # ── 2. Recalcular nível de cada tópico considerando TODOS os indicadores ──
    # (os calculados do formulário + os já existentes na planilha)
    for col_topico, cols_indicadores in MAPA_TOPICOS_IND.items():
        pesos = MAPA_PESOS_IND.get(col_topico, [1] * len(cols_indicadores))
        soma_pond = 0.0
        soma_pesos = 0.0
        for col_nm, peso in zip(cols_indicadores, pesos):
            val = get_cell(col_nm)
            if val is not None:
                soma_pond += val * peso
                soma_pesos += peso
        if soma_pesos > 0:
            nivel_topico = int((soma_pond / soma_pesos) + 0.5)
            set_cell(col_topico, nivel_topico)

    # ── 3. Recalcular nivel de cada dimensao: media ponderada de TODOS os indicadores ──
    # Conforme manual: somatório das frações de contribuição de cada indicador na dimensão
    for col_dim, cols_topicos in MAPA_DIMENSOES_IND.items():
        soma_pond = 0.0
        soma_pesos = 0.0
        for col_top in cols_topicos:
            cols_inds = MAPA_TOPICOS_IND.get(col_top, [])
            pesos_inds = MAPA_PESOS_IND.get(col_top, [1] * len(cols_inds))
            for col_ind, peso in zip(cols_inds, pesos_inds):
                val = get_cell(col_ind)
                if val is not None:
                    soma_pond += val * peso
                    soma_pesos += peso
        if soma_pesos > 0:
            nivel_dim = int((soma_pond / soma_pesos) + 0.5)
            set_cell(col_dim, nivel_dim)

    # ── 4. Nível geral do município (média das 4 dimensões) ───────────────────
    niveis_dims = []
    for col_dim in MAPA_DIMENSOES_IND.keys():
        val = get_cell(col_dim)
        if val is not None:
            niveis_dims.append(val)
    if niveis_dims:
        nivel_geral_final = int((sum(niveis_dims) / len(niveis_dims)) + 0.5)
        set_cell(29, nivel_geral_final)   # col 29 = 'Nível de Maturidade do município'

    # Marcar como avaliada
    ws.cell(row=excel_row, column=4, value='SIM')

    wb.save(arquivo_ind)
    return True


def imprimir_relatorio(dimensoes, municipio_nome, df_ind, linha_idx):
    """Imprime relatório com nível das dimensões recalculado considerando todos os indicadores."""
    html = f'''
    <div style="font-family: Arial, sans-serif; padding: 16px; background: #f8f9fa;
                border-radius: 8px; border-left: 5px solid #1565C0;">
      <h2 style="color:#1565C0; margin-top:0">📊 Resultados — {municipio_nome}</h2>
    '''

    nivel_geral_soma = 0
    dims_calculadas = 0

    CORES_DIM = {
        'Capacidades Institucionais': '#1565C0',
        'Econômica':                  '#2E7D32',
        'Sociocultural':              '#6A1B9A',
        'Meio Ambiente':              '#00695C',
    }

    # Calcular o nivel real de cada dimensão/tópico usando a planilha + formulário
    row_planilha = df_ind.iloc[linha_idx]

    def nm_planilha(col_0based):
        """Retorna o N M de uma coluna da planilha (após salvar indicadores do formulário)."""
        # Já está atualizado na memória do df_ind? Não — mas usamos os dimensoes calculados
        # para os indicadores do formulário e planilha para o resto.
        val = row_planilha.iloc[col_0based]
        try:
            v = float(val)
            return v if v > 0 else None
        except (TypeError, ValueError):
            return None

    for dim_nome, res_dim in dimensoes.items():
        cor = CORES_DIM.get(dim_nome, '#333')

        # Calcular nível real da dimensão com todos os tópicos (planilha + formulário)
        # Para o relatório, mostramos os tópicos calculados pelo formulário com detalhes
        # e informamos que o nível final considera todos os tópicos da planilha
        html += f'''
        <div style="background:white; border-radius:6px; padding:12px;
                    margin:10px 0; border-top: 3px solid {cor};">
          <h3 style="color:{cor}; margin-top:0">{dim_nome}
            <span style="background:{cor}; color:white; border-radius:4px;
                         padding:2px 10px; font-size:1.1em; margin-left:8px">
              Nível {res_dim.nivel_arredondado} *
            </span>
          </h3>
          <p style="color:#888; font-size:0.85em; margin:0 0 8px 0">
            * Nível provisório baseado apenas nos indicadores do formulário.
            O nível final (incluindo todos os indicadores da planilha) será calculado e salvo automaticamente.
          </p>
          <table style="width:100%; border-collapse:collapse; font-size:0.92em">
            <tr style="background:#f0f0f0">
              <th style="text-align:left;padding:4px 8px">Tópico</th>
              <th style="text-align:left;padding:4px 8px">Indicador</th>
              <th style="text-align:center;padding:4px 8px">Letras</th>
              <th style="text-align:center;padding:4px 8px">Pts</th>
              <th style="text-align:center;padding:4px 8px">Peso</th>
              <th style="text-align:center;padding:4px 8px">Nível</th>
            </tr>
        '''
        for top_nome, res_top in res_dim.topicos.items():
            first = True
            n_ind = len(res_top.indicadores)
            for ri in res_top.indicadores:
                rowspan = f'rowspan="{n_ind}"' if first else ''
                top_cell = (f'<td {rowspan} style="border:1px solid #ddd;padding:4px 8px;'
                            f'font-weight:bold;vertical-align:middle">'
                            f'{top_nome}<br><small>Nível {res_top.nivel_arredondado}</small></td>'
                            if first else '')
                letras_str = ', '.join(ri.letras_selecionadas) if ri.letras_selecionadas else '—'
                html += f'''
                <tr>
                  {top_cell}
                  <td style="border:1px solid #ddd;padding:4px 8px">{ri.indicador.nome}</td>
                  <td style="border:1px solid #ddd;padding:4px 8px;text-align:center">{letras_str}</td>
                  <td style="border:1px solid #ddd;padding:4px 8px;text-align:center">{ri.pontuacao_bruta:.0f}</td>
                  <td style="border:1px solid #ddd;padding:4px 8px;text-align:center">{ri.indicador.peso_relevancia}</td>
                  <td style="border:1px solid #ddd;padding:4px 8px;text-align:center;font-weight:bold">{ri.nivel}</td>
                </tr>
                '''
                first = False
        html += '</table></div>'
        nivel_geral_soma += res_dim.nivel_medio
        dims_calculadas += 1

    nivel_geral = nivel_geral_soma / dims_calculadas if dims_calculadas else 0
    nivel_geral_arr = int(nivel_geral + 0.5)

    html += f'''
    <div style="background:#1565C0; color:white; border-radius:6px;
                padding:14px; margin-top:12px; text-align:center">
      <h2 style="margin:0">⭐ Nível Final do Município será recalculado após salvar</h2>
      <small>Os níveis de tópicos e dimensões serão recalculados considerando TODOS os indicadores da planilha</small>
    </div>
    </div>
    '''
    display(HTML(html))
    return nivel_geral


# ══════════════════════════════════════════════════════════════════════════════
# EXECUÇÃO PRINCIPAL
# ══════════════════════════════════════════════════════════════════════════════

# Pegar município selecionado no widget
sel = select_widget.value
idx_selecionado = int(re.match(r'\[(\d+)\]', sel).group(1))
row_resp = df_resp.iloc[idx_selecionado]
municipio_nome = str(row_resp.iloc[2]).strip()

print(f'🏙️  Calculando indicadores para: {municipio_nome}')
print('─' * 60)

# Calcular indicadores do formulário
dimensoes = processar_municipio(row_resp, TODOS_INDICADORES)

# Buscar linha do município na planilha
linha_idx = buscar_linha_municipio(df_ind, municipio_nome)
if linha_idx is None:
    print(f'⚠️  Município "{municipio_nome}" não encontrado na planilha indicadores.')
    linha_idx = 0

# Exibir relatório
nivel_geral = imprimir_relatorio(dimensoes, municipio_nome, df_ind, linha_idx)

# Salvar na planilha (recalculando tópicos e dimensões com TODOS os indicadores)
print()
print('💾 Salvando resultados na planilha indicadores...')
ok = salvar_resultados(dimensoes, municipio_nome, df_ind, INDICADORES_FILE)

if ok:
    print(f'✅ Resultados salvos com sucesso para "{municipio_nome}"!')
    print()
    print('📥 Fazendo download da planilha atualizada...')
    files.download(INDICADORES_FILE)
else:
    print()
    print('ℹ️  Os cálculos foram realizados mas não foram salvos na planilha.')
    print('    Verifique o nome do município e tente novamente.')


🏙️  Calculando indicadores para: Chácara
────────────────────────────────────────────────────────────



💾 Salvando resultados na planilha indicadores...
✅ Resultados salvos com sucesso para "Chácara"!

📥 Fazendo download da planilha atualizada...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>